In [ ]:
import sys
!{sys.executable} --version


import psutil
from functools import partial
 
import joblib
import time
from shutil import copy
import numpy as np
import pandas as pd
#import tensorflow as tf
import os
import matplotlib.pyplot as plt
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm
import pickle
# import xgboost as xgb

from glob import glob
# import psi4
# from helper_CC_ML_spacial import *

import pyscf
from pyscf import gto, scf, mcscf, cc

import ffsim
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 
from qiskit import QuantumCircuit, QuantumRegister
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler

from qiskit_addon_sqd.fermion import SCIResult, diagonalize_fermionic_hamiltonian
from qiskit_addon_sqd.counts import bit_array_to_arrays
from qiskit.primitives import StatevectorSampler, BitArray


from ansatzmap import get_zigzag_physical_layout

from tqdm import tqdm

from DDLUCJ import DDLUCJ, GrabAmps    


In [ ]:
UofT_palette = [ "#1E3765",
                 "#007FA3", 
                 "#6D247A", 
                 "#DC4633",
                 "#6FC7EA",
                 "#00A189",
                 "#AB1368",
                 "#0D534D",
                 "#F1C500",
                 "#8DBF2E"
               ]

palette = sns.color_palette(UofT_palette)

In [ ]:
datadf = pd.read_csv("../../../DDLUCJ_active_spaces_unfrozen.csv",delimiter=';').dropna(axis=1)

In [ ]:
datadf

In [ ]:
dim_df = pd.read_excel("Dimensions.xlsx",index_col=0)

In [ ]:
structure_path_dict = dict(zip(datadf['molecule'], datadf['xyz']))

structure_path_dict = {
    name: os.path.join("../../../classical/structures", xyz)
    for name, xyz in structure_path_dict.items()
}

In [ ]:
moldf = pd.read_csv('molecules.csv')
activespacedf = pd.read_csv("active_spaces.csv")

In [ ]:
# "No recovery" baseline: a single round of hamming-weight postselection on the
# raw quantum samples, diagonalized directly with Fulqrum -- no configuration
# recovery iterations. See DDLUCJ.PostprocessFulqrum(use_recovery=False).
num_batches = 1
samples_per_batch = 1000  # cap on unique half-strings included in the subspace


In [ ]:
checkpoint_path = "Energies_NoRecovery_checkpoint.csv"
# Keyed on (Name, L, Basis, Injection) -- the same identity the filename is
# parsed into below -- rather than the raw filename, since one token in the
# filename (parts[1]) is discarded during parsing and can't be recovered
# from a previously-exported Energies_NoRecovery.xlsx/csv.
checkpoint_cols = ['Name', 'L', 'Basis', 'Injection',
                    'Energy_NoRecovery', 'SubspaceDim_NoRecovery']

def parse_key(path):
    basename = os.path.basename(path)
    parts = basename.replace('.npz', '').split('_')
    name, _, rawlayers, basis = parts[:4]
    injection = '_'.join(parts[4:])
    layer = int(rawlayers.strip("L"))
    return name, layer, basis, injection

# Resume support: load any results from a previous (interrupted) run -- or an
# existing Energies_NoRecovery checkpoint/export you already have -- so we
# don't recompute combinations that already succeeded.
if os.path.exists(checkpoint_path):
    done_df = pd.read_csv(checkpoint_path)
    completed = set(
        done_df[['Name', 'L', 'Basis', 'Injection']].itertuples(index=False, name=None)
    )
    energy_data = list(
        done_df[['Name', 'L', 'Basis', 'Injection',
                 'Energy_NoRecovery', 'SubspaceDim_NoRecovery']].itertuples(index=False, name=None)
    )
    print(f"Resuming from checkpoint: {len(completed)} combination(s) already completed.")
else:
    completed = set()
    energy_data = []
    print("No checkpoint found, starting fresh.")

all_files = sorted(glob("./counts/*npz"))
remaining_files = [f for f in all_files if parse_key(f) not in completed]
print(f"{len(all_files)} total file(s), {len(remaining_files)} remaining to process.")

for i in tqdm(remaining_files):
    basename = os.path.basename(i)
    name, layer, basis, injection = parse_key(i)

    moldict = moldf[moldf['molecule'] == name]
    if moldict.empty:
        print(f"MISSING IN MOLECULES.CSV: {name}")
        continue
    n_electrons = moldict['n_electrons'].values[0]
    num_orbitals = moldict['num_orbitals'].values[0]
    xyzname = moldict['mol_filename'].values[0]
    pathxyz = os.path.join("../../../classical/structures/", xyzname)

    print(f"Running {name}_LUCJ_L{layer}_{basis}_{injection}")

    try:
        ampdict = GrabAmps(name, basis)
        t1, t2 = ampdict[injection]

        dd = DDLUCJ(
            StructurePath=pathxyz,
            BasisSet=basis,
            NElec=int(n_electrons),
            NOrb=int(num_orbitals),
            injected=True,
            t1=t1,
            t2=t2,
            n_reps=int(layer),
            optimization_level=3,
            temp_dir="./",
            clean_temp_dir=True,
            n_jobs=-1,
            num_batches=1,          # see note below
            samples_per_batch=1000,
            verbose=True,
        )

        counts = np.load(i)
        bitstrings = BitArray.from_bool_array(counts['bitstrings'])

        energy, subspace = dd(
            postprocess=True,
            BitArray=bitstrings,
            usefulqrum=True,
            bitarraypath=i,
            use_recovery=False,
        )
        energy = float(np.ravel(energy)[0])
    except Exception as e:
        # Don't let one bad file kill the whole run / lose prior progress
        print(f"FAILED on {basename}: {e!r}")
        continue

    energy_data.append((name, layer, basis, injection, energy, subspace))

    # Write this result to disk immediately so a crash/timeout after this
    # point doesn't lose it -- next run will pick up from here.
    row_df = pd.DataFrame([[name, layer, basis, injection, energy, subspace]],
                          columns=checkpoint_cols)
    write_header = not os.path.exists(checkpoint_path)
    row_df.to_csv(checkpoint_path, mode='a', header=write_header, index=False)
    completed.add((name, layer, basis, injection))

# The ones that fail may actually contain 0 correct configs. 

In [ ]:
# sanity check on the last file processed
name, layers, basis, injection, energy, subspace_dimension = energy_data[-1]
print(f"{name} (L={layers}, {basis}, {injection}): E = {energy:.6f} Ha, subspace dim = {subspace_dimension}")


In [ ]:
norecovery_df = pd.read_csv(checkpoint_path)
norecovery_df.to_excel("Energies_NoRecovery.xlsx", index=False)
norecovery_df